In [2]:
import pandas as pd
import numpy as np

In [4]:
df = pd.read_csv("agricultural_yield_india_cleaned_long.csv")

In [5]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nData types:")
print(df.dtypes)

Shape: (77336, 6)

Columns:
['State', 'District', 'Crop', 'Season', 'Year', 'Yield_Kg_Ha']

Missing values:
State          0
District       0
Crop           0
Season         0
Year           0
Yield_Kg_Ha    0
dtype: int64

Data types:
State           object
District        object
Crop            object
Season          object
Year            object
Yield_Kg_Ha    float64
dtype: object


In [6]:
X = df.drop("Yield_Kg_Ha", axis=1)
y = df["Yield_Kg_Ha"]

In [7]:
print("Features:")
print(X.columns.tolist())

print("\nTarget:")
print(y.name)

Features:
['State', 'District', 'Crop', 'Season', 'Year']

Target:
Yield_Kg_Ha


In [8]:
categorical_features = ["State", "District", "Crop", "Season"]
numerical_features = ["Year"]

print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)

Categorical features: ['State', 'District', 'Crop', 'Season']
Numerical features: ['Year']


In [9]:
print(df["Year"].unique())

['2016-17' '2017-18' '2018-19' '2019-20' '2020-21' '2021-22' '2022-23'
 '2023-24' '2024-25']


In [10]:
train_years = [
    "2016-17", "2017-18", "2018-19",
    "2019-20", "2020-21", "2021-22", "2022-23"
]

test_years = ["2023-24", "2024-25"]

train = df[df["Year"].isin(train_years)].copy()
test = df[df["Year"].isin(test_years)].copy()

print("Training data:", train.shape)
print("Testing data:", test.shape)

Training data: (59984, 6)
Testing data: (17352, 6)


In [20]:
df["Year_Num"] = df["Year"].str[:4].astype(int)

df[["Year", "Year_Num"]].head()

,Year,Year_Num
0,2016-17,2016
1,2016-17,2016
2,2016-17,2016
3,2016-17,2016
4,2016-17,2016


In [21]:
categorical_features = ["State", "District", "Crop", "Season"]
numerical_features = ["Year_Num"]

In [22]:
train = df[df["Year"].isin(train_years)].copy()
test = df[df["Year"].isin(test_years)].copy()

X_train = train.drop("Yield_Kg_Ha", axis=1)
y_train = train["Yield_Kg_Ha"]

X_test = test.drop("Yield_Kg_Ha", axis=1)
y_test = test["Yield_Kg_Ha"]

In [24]:
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("numerical", "passthrough", numerical_features)
    ]
)

In [25]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression

linear_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

print("Linear Regression pipeline created successfully")

Linear Regression pipeline created successfully


In [26]:
linear_model.fit(X_train, y_train)

print("Linear Regression model trained successfully")

Linear Regression model trained successfully


In [27]:
y_pred_linear = linear_model.predict(X_test)

print("Predictions generated:", len(y_pred_linear))
print("First 10 predictions:")
print(y_pred_linear[:10])

Predictions generated: 17352
First 10 predictions:
[2034.56169762 2464.83069819  249.92144895 1971.82457508  398.16478562
  368.90833492 2398.3714299   183.46218066 1905.36530679  302.44906663]


In [28]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

mae_linear = mean_absolute_error(y_test, y_pred_linear)
rmse_linear = np.sqrt(mean_squared_error(y_test, y_pred_linear))
r2_linear = r2_score(y_test, y_pred_linear)

print("Linear Regression Results")
print("MAE :", mae_linear)
print("RMSE:", rmse_linear)
print("R²  :", r2_linear)

Linear Regression Results
MAE : 604.7813927424281
RMSE: 3152.653317528652
R²  : 0.09755512150476564


In [31]:
from sklearn.ensemble import RandomForestRegressor

rf_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

print("Random Forest pipeline created successfully")

Random Forest pipeline created successfully


In [33]:
rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully")

Random Forest model trained successfully


In [34]:
y_pred_rf = rf_model.predict(X_test)

print("Predictions generated:", len(y_pred_rf))
print("First 10 predictions:")
print(y_pred_rf[:10])

Predictions generated: 17352
First 10 predictions:
[1579.96 1991.99  371.94 1980.39  542.49  376.79 1991.99  496.94 1968.97
  354.55]


In [35]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest Results")
print("MAE :", mae_rf)
print("RMSE:", rmse_rf)
print("R²  :", r2_rf)

Random Forest Results
MAE : 350.2418505071461
RMSE: 3108.3639707081998
R²  : 0.1227326114287377


In [39]:
from xgboost import XGBRegressor

xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBRegressor(
        n_estimators=100,
        max_depth=6,
        learning_rate=0.1,
        random_state=42,
        n_jobs=-1
    ))
])

print("XGBoost pipeline created successfully")

XGBoost pipeline created successfully


In [40]:
xgb_model.fit(X_train, y_train)

print("XGBoost model trained successfully")

XGBoost model trained successfully


In [41]:
y_pred_xgb = xgb_model.predict(X_test)

print("Predictions generated:", len(y_pred_xgb))
print("First 10 predictions:")
print(y_pred_xgb[:10])

Predictions generated: 17352
First 10 predictions:
[2288.5095 2454.3271  732.6478 2288.5095 1004.9549  771.1962 2454.3271
  732.6478 2288.5095  771.1962]


In [42]:
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
rmse_xgb = np.sqrt(mean_squared_error(y_test, y_pred_xgb))
r2_xgb = r2_score(y_test, y_pred_xgb)

print("XGBoost Results")
print("MAE :", mae_xgb)
print("RMSE:", rmse_xgb)
print("R²  :", r2_xgb)

XGBoost Results
MAE : 459.61763486954516
RMSE: 3137.3737977140813
R²  : 0.10628142826948173
